In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')


In [3]:
from hotelling.spatial.census import build_grid_polygons

grid = build_grid_polygons(grid)
grid['index'] = grid.index


In [4]:
# Exploration: view raw OSM station data (already cached after first run)
# The identify_transport_hubs() function uses this cache internally.
from hotelling.spatial.osm import fetch_pois

osm_stations = fetch_pois(type="stations", city="Berlin")
print(f"OSM stations: {len(osm_stations)} rows")
osm_stations.head()


OSM stations: 283 rows


,osm_id,osm_type,geometry,contact:website,light_rail,name,network,network:short,network:wikidata,official_name,...,description:old_name,source:old_name,internet_access:ssid,shelter,name:ko,name:ru,railway:ref:BVG,ref_name,old_name1,point
0,21302157,node,POINT (13.33645 52.51437),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Tiergarten,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Tiergarten,...,None,None,None,None,None,None,None,None,None,POINT (13.33645 52.51437)
1,26124376,node,POINT (13.28442 52.51801),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Westend,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Westend,...,None,None,None,None,None,None,None,None,None,POINT (13.28442 52.51801)
2,26603219,node,POINT (13.17981 52.42141),None,yes,Wannsee,Verkehrsverbund Berlin-Brandenburg,VBB,Q315451,Berlin-Wannsee S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.17981 52.42141)
3,26943869,node,POINT (13.22763 52.51015),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Pichelsberg,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin Pichelsberg,...,None,None,None,None,None,None,None,None,None,POINT (13.22763 52.51015)
4,27215654,node,POINT (13.26101 52.48825),http://www.s-bahn-berlin.de/fahrplanundnetz/ba...,yes,Grunewald,Verkehrsverbund Berlin-Brandenburg,VBB,None,Berlin-Grunewald S-Bahn,...,None,None,None,None,None,None,None,None,None,POINT (13.26101 52.48825)


In [5]:
from hotelling.spatial.city_data import identify_transport_hubs

grid_with_stations = identify_transport_hubs(grid)

print(f"Cells with at least one station: {(grid_with_stations['station_count'] > 0).sum()}")
print(f"Cells with DB station class:     {grid_with_stations['station_class'].notna().sum()}")
print("Station class distribution:")
print(grid_with_stations['station_class'].value_counts().sort_index().to_string())


Cells with at least one station: 161
Cells with DB station class:     62
Station class distribution:
station_class
1.0     8
2.0     7
3.0     9
4.0    36
5.0     2


In [6]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    # Plot the grid colored by station class
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    grid_with_stations.plot(column='station_class', ax=ax, cmap='tab10', edgecolor='none', legend=True, alpha=1)
    grid_with_stations.plot(ax=ax, color = 'firebrick', linewidth=1, alpha=0.2)
    berlin.plot(ax=ax, edgecolor='royalblue', color='none', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    plt.title("Grid with DB Station Classes")
    plt.show()


In [7]:
# grid_with_stations.parquet saved automatically by identify_transport_hubs()
print(f"grid_with_stations: {len(grid_with_stations)} cells")
print(f"Output: data/processed/grid_with_stations.parquet")


grid_with_stations: 16170 cells
Output: data/processed/grid_with_stations.parquet


In [8]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_04_transport_hubs.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")

Outputs cleared: GEO_04_transport_hubs.ipynb
